# BERT — Pre-training of Deep Bidirectional Transformers (2018)

_Papers · Transformers & LLMs_

**Pre-train a Transformer encoder by masking 15% of the words and predicting them from BOTH sides, then fine-tune the same model on any language task.**

---

This notebook is a practice scaffold for the **BERT — Pre-training of Deep Bidirectional Transformers (2018)** lesson. We build it step by step: the cross-entropy at a single masked slot, a tiny corpus, the attention and Transformer blocks, the 15% masking scheme, the training loop, and finally a causal-attention ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We assemble a tiny BERT in six steps: (1) the masked-LM loss at one slot, (2) a small corpus and vocabulary, (3) positional encoding, (4) bidirectional self-attention and a Transformer block, (5) the 15% MLM masking scheme, and (6) the training loop plus a causal-attention ablation.

### Step 1 — The masked-LM loss at a single slot

BERT's pre-training target is to predict the original word at each masked position. At one slot the model emits a vector of logits over the vocabulary; softmax turns them into probabilities, and the loss is `-log p(true word)`. Here we hand-compute it for a 6-word vocabulary where the true masked word is index 2 ("sat"), and confirm it matches `F.cross_entropy`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Worked example: cross-entropy at ONE masked slot over a V=6 vocabulary.
# Vocab indices: [the, cat, sat, dog, ran, mat]; true masked word is index 2 ("sat").
logits = torch.tensor([0.5, 1.0, 3.0, 0.2, -1.0, 0.8])
target = 2

p = F.softmax(logits, dim=0)
loss_by_hand = -math.log(p[target].item())
loss_builtin = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])).item()

print("softmax probs =", [round(v, 4) for v in p.tolist()])   # [0.0583, 0.0962, 0.7106, ...]
print("p(true='sat') =", round(p[target].item(), 4))           # 0.7106
print("MLM loss = -log p(true) =", round(loss_by_hand, 4))     # 0.3417
print("F.cross_entropy        =", round(loss_builtin, 4))      # 0.3417

### Step 2 — A tiny corpus and vocabulary

We use eight short sentences and build a vocabulary from the words they contain, reserving two special tokens: `[PAD]` for padding and `[MASK]` for the slots BERT will hide. Each sentence becomes a row of integer token ids in the `data` tensor.

In [ ]:
# Tiny word corpus. Reserve [PAD] and [MASK].
sentences = ["the cat sat on the mat", "the dog sat on the log",
             "the cat ran on the mat", "the dog ran on the log",
             "a cat sat on a mat",     "a dog sat on a log",
             "the cat sat on the log", "the dog ran on the mat"]

words = sorted(set(" ".join(sentences).split()))
PAD, MASK = "[PAD]", "[MASK]"
vocab = [PAD, MASK] + words
stoi = {w: i for i, w in enumerate(vocab)}

V = len(vocab)
SEQ = 6
data = torch.tensor([[stoi[w] for w in s.split()] for s in sentences])   # (N, SEQ)
print("vocab (V=%d):" % V, vocab)

### Step 3 — Positional encoding

A Transformer has no built-in sense of word order, so we add the sinusoidal positional encoding from the original Transformer paper. Even positions use sine, odd positions use cosine, with the frequency decreasing across the embedding dimension. This gives every position a unique, fixed signature added to its word embedding.

In [ ]:
# Positional encoding (from paper-transformer).
def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    denom = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)
    pe[:, 1::2] = torch.cos(pos / denom)
    return pe

### Step 4 — Bidirectional self-attention and a Transformer block

`MHA` is multi-head self-attention. The key flag is `causal`: when `False` (BERT's default) every position may attend to **every** other position, so a masked word sees context from both sides. When `True` it masks out future positions — that is the left-to-right ablation we run later. `Block` wraps attention and a GELU feed-forward network with residual connections and LayerNorm.

In [ ]:
# Multi-head self-attention. causal=False -> BIDIRECTIONAL (BERT); True -> the ablation.
class MHA(nn.Module):
    def __init__(self, d, h, causal=False):
        super().__init__()
        self.h, self.dk, self.causal = h, d // h, causal
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))

    def split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)

    def forward(self, x):
        Q = self.split(self.Wq(x))
        K = self.split(self.Wk(x))
        Vv = self.split(self.Wv(x))
        S = x.shape[1]
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if self.causal:                                  # left-to-right: forbid attending to the right
            cm = torch.triu(torch.ones(S, S), diagonal=1).bool()
            scores = scores.masked_fill(cm, float('-inf'))
        a = F.softmax(scores, dim=-1) @ Vv
        B, _, S, _ = a.shape
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))


class Block(nn.Module):
    def __init__(self, d, h, ff, causal=False):
        super().__init__()
        self.a = MHA(d, h, causal)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))   # GELU: BERT's activation
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):
        x = self.n1(x + self.a(x))
        return self.n2(x + self.f(x))


# Tiny BERT: embed + position -> L bidirectional blocks -> MLM head (d_model -> V).
class TinyBERT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=128, L=2, mx=SEQ, causal=False):
        super().__init__()
        self.e = nn.Embedding(V, d)
        self.register_buffer("pe", positional_encoding(mx, d))
        self.b = nn.ModuleList([Block(d, h, ff, causal) for _ in range(L)])
        self.head = nn.Linear(d, V)                      # MLM head: logits over the vocabulary per position

    def forward(self, t):
        x = self.e(t) + self.pe[:t.shape[1]]
        for blk in self.b:
            x = blk(x)
        return self.head(x)                              # (B, S, V)

### Step 5 — The 15% MLM masking scheme

BERT chooses 15% of positions to predict. Of those, 80% are replaced with `[MASK]`, 10% with a random word, and 10% left unchanged (Sec 3.1) — the 80/10/10 split keeps the model from keying off the `[MASK]` symbol, which is absent at fine-tune time. Labels are -100 (ignored by cross-entropy) everywhere except the chosen slots, where the label is the original word. On this tiny data we guarantee at least one masked token per row.

In [ ]:
# MLM masking: 15% of positions; of those 80% -> [MASK], 10% -> random, 10% -> unchanged (Sec 3.1).
def mask_batch(batch):
    inp = batch.clone()
    labels = torch.full_like(batch, -100)                # -100 = ignored by cross_entropy
    chosen = torch.rand(batch.shape) < 0.15

    for r in range(batch.shape[0]):                      # tiny data: guarantee >=1 masked token per row
        if not chosen[r].any():
            chosen[r, torch.randint(0, batch.shape[1], (1,))] = True

    labels[chosen] = batch[chosen]                       # predict the ORIGINAL word at chosen slots

    r2 = torch.rand(batch.shape)
    inp[chosen & (r2 < 0.8)] = stoi[MASK]                # 80% -> [MASK]
    rand = chosen & (r2 >= 0.8) & (r2 < 0.9)             # 10% -> random word
    inp[rand] = torch.randint(2, V, (int(rand.sum()),))
    # remaining 10% -> unchanged (already a copy)
    return inp, labels

### Step 6 — Train, fill a mask, then run the causal ablation

Now we train the bidirectional model on the MLM objective and watch the loss fall. `fill_mask` hides one word and reads the top predictions, confirming the model learned sensible fills. Finally the ablation: retrain with `causal=True` so each masked position loses its right-hand context, and compare the final losses — bidirectional should reach a clearly lower loss.

In [ ]:
# Train on the MLM objective; print the loss FALLING.
def train(causal=False, steps=600, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    net = TinyBERT(V, causal=causal)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    losses = []
    for s in range(steps):
        inp, labels = mask_batch(data)
        loss = F.cross_entropy(net(inp).reshape(-1, V), labels.reshape(-1), ignore_index=-100)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if s % 100 == 0 or s == steps - 1:
            print(f"  step {s:4d}  MLM loss {loss.item():.4f}")
    return net, losses

print("TRAIN tiny BERT (bidirectional MLM):")
net, losses = train(causal=False)


# Fill a [MASK]: feed a sentence with one slot masked, read the top predictions.
def fill_mask(net, sent, pos):
    toks = [stoi[w] for w in sent.split()]
    toks[pos] = stoi[MASK]
    with torch.no_grad():
        probs = F.softmax(net(torch.tensor([toks]))[0, pos], dim=0)
    top = torch.topk(probs, 3)
    return [(vocab[i], round(probs[i].item(), 3)) for i in top.indices.tolist()]

print('\nFILL THE [MASK]:')
print('  "the cat [MASK] on the mat" ->', fill_mask(net, "the cat sat on the mat", 2))   # 'sat' wins
print('  "the [MASK] sat on the log" ->', fill_mask(net, "the dog sat on the log", 1))   # 'dog' wins


# ABLATION: causal (left-to-right) attention -> the masked word loses its RIGHT context.
print("\nABLATION: causal (left-to-right) instead of bidirectional:")
_, losses_causal = train(causal=True)
print(f"\nfinal MLM loss  bidirectional: {losses[-1]:.4f}   causal: {losses_causal[-1]:.4f}")
# Bidirectional reaches a clearly lower (better) loss: it can use both sides to fill the blank.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does the masked-language-model loss fall as we train a tiny bidirectional BERT, and does removing bidirectionality (a causal/left-to-right ablation) leave the loss higher?_

### Step 1 — Rebuild the tiny BERT and the training step

This visualization cell is self-contained, so it re-imports torch and re-defines the same corpus, positional encoding, attention, block, TinyBERT, masking, and a one-run training function from the reference implementation. Nothing changes — it just reproduces the machinery so we can average several runs next.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

sentences = ["the cat sat on the mat", "the dog sat on the log",
             "the cat ran on the mat", "the dog ran on the log",
             "a cat sat on a mat",     "a dog sat on a log",
             "the cat sat on the log", "the dog ran on the mat"]
words = sorted(set(" ".join(sentences).split()))
vocab = ["[PAD]", "[MASK]"] + words
stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
data = torch.tensor([[stoi[w] for w in s.split()] for s in sentences])


def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    denom = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)
    pe[:, 1::2] = torch.cos(pos / denom)
    return pe


class MHA(nn.Module):
    def __init__(self, d, h, causal):
        super().__init__()
        self.h, self.dk, self.causal = h, d // h, causal
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))

    def split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)

    def forward(self, x):
        Q = self.split(self.Wq(x))
        K = self.split(self.Wk(x))
        Vv = self.split(self.Wv(x))
        S = x.shape[1]
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if self.causal:
            sc = sc.masked_fill(torch.triu(torch.ones(S, S), 1).bool(), float('-inf'))
        a = F.softmax(sc, dim=-1) @ Vv
        B, _, S, _ = a.shape
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))


class Block(nn.Module):
    def __init__(self, d, h, ff, causal):
        super().__init__()
        self.a = MHA(d, h, causal)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):
        x = self.n1(x + self.a(x))
        return self.n2(x + self.f(x))


class TinyBERT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=128, L=2, mx=6, causal=False):
        super().__init__()
        self.e = nn.Embedding(V, d)
        self.register_buffer("pe", positional_encoding(mx, d))
        self.b = nn.ModuleList([Block(d, h, ff, causal) for _ in range(L)])
        self.head = nn.Linear(d, V)

    def forward(self, t):
        x = self.e(t) + self.pe[:t.shape[1]]
        for blk in self.b:
            x = blk(x)
        return self.head(x)


def mask_batch(batch):
    inp = batch.clone()
    labels = torch.full_like(batch, -100)
    chosen = torch.rand(batch.shape) < 0.15
    for r in range(batch.shape[0]):
        if not chosen[r].any():
            chosen[r, torch.randint(0, batch.shape[1], (1,))] = True
    labels[chosen] = batch[chosen]
    r2 = torch.rand(batch.shape)
    inp[chosen & (r2 < 0.8)] = stoi["[MASK]"]
    rd = chosen & (r2 >= 0.8) & (r2 < 0.9)
    inp[rd] = torch.randint(2, V, (int(rd.sum()),))
    return inp, labels


def run(causal, steps=600, seed=0):
    torch.manual_seed(seed)
    net = TinyBERT(V, causal=causal)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    L = []
    for s in range(steps):
        inp, lab = mask_batch(data)
        loss = F.cross_entropy(net(inp).reshape(-1, V), lab.reshape(-1), ignore_index=-100)
        opt.zero_grad()
        loss.backward()
        opt.step()
        L.append(loss.item())
    return L

### Step 2 — Average several seeds and compare the two loss curves

A single run is noisy on data this small, so we average four seeds for each setting to get a clean curve. We print the loss at regular steps for the bidirectional model and the causal one side by side: the bidirectional curve should settle clearly lower, because it can use both sides of the blank to predict it.

In [ ]:
import statistics

# Average over seeds for a cleaner curve.
def avg(causal, seeds=(0, 1, 2, 3)):
    runs = [run(causal, seed=s) for s in seeds]
    return [statistics.mean(col) for col in zip(*runs)]

bi = avg(False)
ca = avg(True)

idx = list(range(0, 600, 50)) + [599]
print("bidirectional:", [[i, round(bi[i], 3)] for i in idx])
print("causal       :", [[i, round(ca[i], 3)] for i in idx])
# bidirectional -> ~0.28 (uses both sides). causal -> ~0.66 (right context hidden). Our small run, not the paper's.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your tiny BERT drives the masked-LM loss down with full bidirectional
            attention. Now switch the attention to causal (each position may only attend to positions at
            or before it), keep everything else identical, and retrain. What happens to the final MLM loss, and
            what does that show about why BERT reads both directions?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: add a causal mask inside attention so position $i$ cannot attend to positions $\gt i$. Keep the corpus, the 15% masking, depth, width, heads, optimizer, and seed identical. — _An honest ablation changes only the variable under test &mdash; here, bidirectional vs. left-to-right &mdash; so any difference is attributable to it._
- Retrain and compare the final MLM loss. In our run the bidirectional model reaches ~0.28 while the causal one plateaus higher, around ~0.66 (averaged over seeds). — _To fill "the cat ___ on the mat", the right context ("on the mat") is informative; a left-to-right model at the blank cannot see it, so it predicts less well and the loss stays higher._
- Conclude that the two-sided context, not extra capacity, is what lets BERT fill masks better. — _Both runs share architecture, parameter count, data, and seed; only the attention direction differs, isolating bidirectionality as the cause._

**Answer:** With causal (left-to-right) attention the masked-LM loss settles at a clearly higher
                 (worse) value than the bidirectional model (~0.66 vs ~0.28 in our small run). Because the only
                 difference between the two runs is whether a masked position may read the words to its right,
                 this isolates bidirectionality as the reason BERT fills blanks better &mdash; exactly
                 the limitation of left-to-right models that &sect;3.1's masked objective was designed to
                 remove. The CODEVIZ panel shows the two loss curves.

</details>

**Problem 2.** In the worked example the head's logits at the masked slot were $[0.5, 1.0, 3.0, 0.2, -1.0, 0.8]$
            and the true word was index 2 ("sat"). We got loss $0.3417$. Suppose instead the head had been
            certain and wrong &mdash; logit $5.0$ on index 4 ("ran") and small logits elsewhere &mdash;
            would the loss be larger or smaller, and why does that push the model in the right direction?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the loss reads only the true word's probability: $-\log \hat{p}_i[x_i]$ with $x_i = 2$. — _Cross-entropy ignores how mass is split among the wrong words; only the probability on the true word matters._
- A huge logit on the wrong word (index 4) drains probability away from index 2, so $\hat{p}_i[2]$ becomes tiny, and $-\log$ of a tiny number is large. — _Confident-and-wrong is exactly the case cross-entropy punishes hardest._
- Gradient descent on that large loss raises the logit for the true word and lowers the others next time. — _That is how the MLM objective teaches the encoder to put mass on the actually-masked word._

**Answer:** Larger. The loss depends only on the probability the model gives the true word
                 ("sat", index 2). Pouring confidence onto the wrong word ("ran") starves index 2 of
                 probability, so $\hat{p}_i[2]$ is tiny and $-\log \hat{p}_i[2]$ is big &mdash; far above
                 $0.3417$. The large gradient then pushes the head to raise the true word's logit and lower the
                 others, which is precisely how masked prediction trains useful two-sided representations.

</details>

**Problem 3.** The paper masks only 15% of tokens, and of those uses [MASK] just 80% of
            the time (10% random word, 10% unchanged). Why not mask far more tokens, and why not use
            [MASK] 100% of the time for the chosen ones?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Think about masking, say, 90% of tokens. — _With almost nothing left as context, there is too little signal on each side to predict the blanks &mdash; the task becomes near-impossible and pre-training is inefficient._
- Think about always using [MASK] for chosen tokens. — _The [MASK] symbol never appears at fine-tune time, so a model that only ever predicts under it sees a different input distribution downstream &mdash; a pre-train/fine-tune mismatch (\S 3.1)._
- Note the 10%-random and 10%-unchanged force the model to build a good representation of every token, since it cannot tell which positions were chosen. — _It must stay ready to "correct or confirm" any token, not just obvious [MASK] slots._

**Answer:** 15% balances two pressures: enough masked targets to learn from, but enough surrounding
                 context left intact to make each prediction solvable &mdash; masking most of the sentence would
                 starve the context. The 80/10/10 split (&sect;3.1) fixes a distribution mismatch:
                 because [MASK] is absent at fine-tune time, sometimes substituting a random word
                 or keeping the original forces the model to encode every token usefully rather than keying off
                 the [MASK] symbol, so the pre-trained encoder transfers cleanly.

</details>